In [20]:
#####################################################################################################################################
##### This script evaluates the performance of the LemonIte network against PKN
##### This will be done in two different approaches:
#####       1. Classic evaluation metrics (accuracy, precision, recall, F1-score) of LemonIte network against the ground truth data
#####       2. Evaluation of average shortest paths for metabolite-gene pairs from LemonIte network in the PKN
#####################################################################################################################################

In [21]:
# import modules and set working directory
import os
import pandas as pd
import numpy as np
import networkx as nx
from multiprocessing import Pool
from scipy.stats import hypergeom
from adjustText import adjust_text
import matplotlib.pyplot as plt
import seaborn as sns

os.chdir('/home/borisvdm/Documents/PhD/thesis_Mirte/Wang2021/results/LemonTree/noProteomics_percentile2/')

percentile = 2
n_modules = '46'

workdir = os.getcwd()


dataset_file = workdir + f'/Preprocessing/LemonPreprocessed_complete.txt'
lemonnetwork_file = workdir + f'/Networks/LemonNetwork_percentile{percentile}_{n_modules}modules.txt'
PKNnetwork_file = '/home/borisvdm/Documents/PhD/Lemonite/PKN/LemonIte_PKN.tsv'
annotated_mets = workdir+ f'/Preprocessing/name_map.csv' # paste metabolite list in MetaboAnalyst id mapping tool and download result + manually look into annotations
#metabolite_interactions_file = workdir + f'/PKN/HMDB_metabolites_gene_interactions.tsv'


TF_regulator_file = workdir+'/ModuleViewer_files/Lovering.percentile'+str(percentile)+'_list.txt'
metabolite_regulator_file = workdir+'/ModuleViewer_files/Metabolite.percentile'+str(percentile)+'_list.txt'
#metabolite_random_regulator_file = workdir+'ModuleViewer_files/Metabolite.randomreg_list.txt'
clusterfile = workdir+'/Lemon_out/clusters_list.txt'
metabolite_interactions_file = '/home/borisvdm/Documents/PhD/Lemonite/PKN/HMDB_metabolites_gene_interactions.csv'


# Read predicted network and create some dictionaries to map modules to genes, regulators to genes, etc

In [22]:
lemonnetwork = pd.read_csv(lemonnetwork_file, sep='\t')
lemonnetwork
lemonnetwork_metabolites = lemonnetwork[lemonnetwork['Type'] == 'Metabolite-gene']
print(lemonnetwork_metabolites)
# create dict with metabolite-gene interactions
metabolites2genes = {}
for index, row in lemonnetwork_metabolites.iterrows():
    if row['Regulator'] not in metabolites2genes:
        metabolites2genes[row['Regulator']] = [row['Target']]
    else:
        metabolites2genes[row['Regulator']].append(row['Target'])

genes_in_dataset = lemonnetwork['Target'].unique()
print(f'There are {len(genes_in_dataset)} genes in the dataset')

        Regulator  Target      Score  Lemon_module             Type
9969   creatinine  VSTM2L  64.743192             0  Metabolite-gene
9970   creatinine  GABRA5  64.743192             0  Metabolite-gene
9971   creatinine    SYT4  64.743192             0  Metabolite-gene
9972   creatinine   CREG2  64.743192             0  Metabolite-gene
9973   creatinine  SYNGR3  64.743192             0  Metabolite-gene
...           ...     ...        ...           ...              ...
14982  L_cysteine   NXPH3  13.740352            62  Metabolite-gene
14983  L_cysteine  COL9A1  13.740352            62  Metabolite-gene
14984  L_cysteine    GAD1  13.740352            62  Metabolite-gene
14985  L_cysteine  FRMPD1  13.740352            62  Metabolite-gene
14986  L_cysteine  SMIM18  13.740352            62  Metabolite-gene

[5018 rows x 5 columns]
There are 1930 genes in the dataset


In [23]:
# Funtion to create a dictionary with all regulators per module (start from the regulators.fold file created using LemonTree_downstream.py)
def get_regulators(regfile):
    regs = {}
    with open(regfile) as f:
        for line in f:
            module = line.split('\t')[0]
            regulators = line.rstrip().split('\t')[1].split('|')
            regs[module] = regulators
    return regs

module2TFs = get_regulators(TF_regulator_file)
module2mets = get_regulators(metabolite_regulator_file)
module2genes = get_regulators(clusterfile)
#module2mets_random = get_regulators(metabolite_random_regulator_file)

print('modules:', module2genes)
print('TFs:', module2TFs)
print('mets:', module2mets)
#print('mets_random:', module2mets_random)


modules: {'0': ['VSTM2L', 'GABRA5', 'SYT4', 'CREG2', 'SYNGR3', 'RBFOX1', 'KCNS1', 'SYP', 'GPR83', 'PRSS3', 'STX1B', 'TGFBR3L', 'GLT1D1', 'AK5', 'SPTB', 'C4orf50', 'VIPR1', 'TMEM130', 'IQSEC3', 'SYN1', 'CHD5', 'GREM2', 'SYT7', 'PRMT8', 'KCNJ4', 'HPCAL4', 'GABRD', 'ADAM11', 'EPHB6', 'KCTD16', 'GALNTL6', 'RASAL1', 'SLC17A7', 'CPLX1', 'DOC2A', 'GPR61', 'PACSIN1', 'GABRB2', 'PCP4L1', 'ARHGDIG', 'TMEM155', 'CABP1', 'CAMK2A', 'NRGN', 'DLGAP3', 'SNAP25', 'DDN', 'MTUS2', 'CACNA1B', 'GNG3', 'MPPED1', 'SLC6A17', 'SNCB', 'SLC6A7', 'SLC8A2', 'VWA5B2', 'PPP2R2C', 'SV2C', 'CALY', 'SLC30A3', 'SLC12A5', 'HPCA', 'NEFM', 'MAS1', 'CPNE6', 'C1QTNF4', 'UNC5A', 'AJAP1', 'SYT5', 'MFSD4', 'STYK1', 'GLS2', 'KIAA1644', 'ICAM5', 'ATP8A2', 'NEURL1', 'KIAA1045', 'ATP2B3', 'SYN2', 'CAMK1G', 'CRYM', 'MAP7D2', 'PHYHIP', 'FSTL4', 'TRHDE', 'RTN4RL1', 'PSD', 'RYR2'], '1': ['GAD2', 'CACNG3', 'GJD2', 'NWD2', 'PCSK2', 'SYNPR', 'GABRG2', 'WSCD2', 'FADS6', 'GABRG3', 'KLHL1', 'CCK', 'GPR26', 'MYT1L', 'EPHA6', 'LRTM2', 'HCN1', 

In [24]:
# How many unique metabolites in module2mets.values()?
module2mets_values = [item for sublist in module2mets.values() for item in sublist]
print(f'There are {len(set(module2mets_values))} unique metabolites in module2mets')

There are 53 unique metabolites in module2mets


# Subset metabolite-gene interactions file for metabolites that are in the dataset

In [25]:
interactions = pd.read_csv(metabolite_interactions_file, sep='\t')
interactions

/tmp/ipykernel_78145/3026841620.py:1: DtypeWarning: Columns (15,16,17,18,19,20) have mixed types. Specify dtype option on import or set low_memory=False.
  interactions = pd.read_csv(metabolite_interactions_file, sep='\t')


,Name,HMDB,ChEBI,KEGG,PubChem,IUPAC_Name,SMILES,InChIKey,PDB_ID,Kingdom,...,IntAct,UniProtKB,chEMBL,LINCS,STITCH,BioGRID,Human1_GEM_dist1,Human1_GEM_dist2,MetalinksDB,All_database_interactions
0,1-Methylhistidine,HMDB0000001,50599.0,C01152,92105.0,(2S)-2-amino-3-(1-methyl-1H-imidazol-4-yl)prop...,CN1C=NC(C[C@H](N)C(O)=O)=C1,BRMWTNUJHUMWMS-LURJTMIESA-N,NaN,Organic acids and derivatives,...,none,none,none,NaN,NaN,ACTA1,NaN,NaN,NaN,ACTA1
1,"1,3-Diaminopropane",HMDB0000002,15725.0,C00986,428.0,"propane-1,3-diamine",NCCCN,XFNJVJPLKCPIBV-UHFFFAOYSA-N,NaN,Organic nitrogen compounds,...,none,DHPS|DS|DHPS|DHPS|DHPS|DHPS|DHPS|DHPS,none,NaN,NaN,NaN,AOC1|AOC2|AOC3|DHPS|SLC22A1|SLC22A2|SLC22A3,AADAC|AASDH|ABHD12|ABHD6|ACAA1|ACAA2|ACE|ACE2|...,NaN,DHPS|DS|DHPS|DHPS|DHPS|DHPS|DHPS|DHPS|AOC1|AOC...
2,2-Ketobutyric acid,HMDB0000005,30831.0,C00109,58.0,2-oxobutanoic acid,CCC(=O)C(O)=O,TYEYBOSBBBHJIV-UHFFFAOYSA-N,NaN,Organic acids and derivatives,...,none,RIDA|HRSP12|SDS|SDH|SDSL|GOT1|CTH|AGXT2|AGT2|A...,none,NaN,NaN,NaN,BCKDHA|BCKDHB|CTH|GPT|LDHAL6A|LDHAL6B|LDHB|LDH...,AADAC|AADAT|ABAT|ABHD12|ABHD6|ACE|ACE2|ACHE|AC...,NaN,RIDA|HRSP12|SDS|SDH|SDSL|GOT1|CTH|AGXT2|AGT2|A...
3,2-Hydroxybutyric acid,HMDB0000008,50613.0,C05984,440864.0,(2S)-2-hydroxybutanoic acid,CC[C@H](O)C(O)=O,AFENDNXGAFYKQO-VKHMYHEASA-N,NaN,Organic acids and derivatives,...,none,none,none,NaN,NaN,NaN,NaN,NaN,SLC16A6|LDHC|SLC16A3|LDHB|LDHAL6B|SLC16A7|SLC1...,SLC16A6|LDHC|SLC16A3|LDHB|LDHAL6B|SLC16A7|SLC1...
4,2-Methoxyestrone,HMDB0000010,1189.0,C05299,440624.0,"(1S,10R,11S,15S)-5-hydroxy-4-methoxy-15-methyl...",[H][C@@]12CCC(=O)[C@@]1(C)CC[C@]1([H])C3=C(CC[...,WHEUWNKSCXYKBU-QPWUGHHJSA-N,NaN,Lipids and lipid-like molecules,...,none,COMT|UGT1A8|GNT1|UGT1|COMT|COMT|COMT|COMT|COMT,none,HSD17B1,NaN,NaN,ABCB1|COMT|LRTOMT,AADAC|ABCB1|ABHD12|ABHD6|ACE|ACE2|ACHE|ACP2|AC...,UGT1A8|ABCB1|COMT,COMT|UGT1A8|GNT1|UGT1|COMT|COMT|COMT|COMT|COMT...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
217915,Nordeoxycholic acid,HMDB0304947,NaN,NaN,314374.0,"3-{5,16-dihydroxy-2,15-dimethyltetracyclo[8.7....",CC(CC(O)=O)C1CCC2C3CCC4CC(O)CCC4(C)C3CC(O)C12C,PLRQOCVIINWCFA-UHFFFAOYSA-N,NaN,Lipids and lipid-like molecules,...,none,none,none,NaN,NaN,NaN,NaN,NaN,NaN,NaN
217916,3-Oxo-5beta-cholanoic acid,HMDB0304950,NaN,NaN,543448.0,"4-{2,15-dimethyl-5-oxotetracyclo[8.7.0.0^{2,7}...",CC(CCC(O)=O)C1CCC2C3CCC4CC(=O)CCC4(C)C3CCC12C,KIQFUORWRVZTHT-UHFFFAOYSA-N,NaN,Lipids and lipid-like molecules,...,none,DHRS4|SDR25C2|UNQ851/PRO1800,none,NaN,NaN,NaN,NaN,NaN,NaN,DHRS4|SDR25C2|UNQ851/PRO1800
217917,Glycerol 1-myristate,HMDB0304951,75562.0,NaN,79050.0,"2,3-dihydroxypropyl tetradecanoate",CCCCCCCCCCCCCC(=O)OCC(O)CO,DCBSHORRWZKAKO-UHFFFAOYSA-N,NaN,Lipids and lipid-like molecules,...,none,ENPP6|UNQ1889/PRO4334|EPHX2|ABHD6|MGLL|MOGAT1|...,none,NaN,NaN,NaN,NaN,NaN,NaN,ENPP6|UNQ1889/PRO4334|EPHX2|ABHD6|MGLL|MOGAT1|...
217918,O-Phenolsulfonic acid,HMDB0304953,71049.0,NaN,11867.0,2-hydroxybenzene-1-sulfonic acid,OC1=CC=CC=C1S(O)(=O)=O,IULJSGIJJZZUMF-UHFFFAOYSA-N,NaN,Benzenoids,...,none,none,none,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
# contextualize metabolite-gene interactions to dataset
metabolite_mapping = pd.read_csv(annotated_mets, sep=',')
print(metabolite_mapping)
# Create a dictionary that maps Query to HMDB
metabolite_mapping = metabolite_mapping.set_index('Query')['HMDB'].dropna().to_dict()
print(metabolite_mapping)
metabolites_hmdb = metabolite_mapping.values()
print(f'{len(metabolites_hmdb)} metabolites in this dataset are mapped to a hmdb id')

                      Query                   Match         HMDB   PubChem  \
0     2_hydroxybutyric_acid   2-Hydroxybutyric acid  HMDB0000008  440864.0   
1    2_hydroxyglutaric_acid      2-Hydroxyglutarate  HMDB0059655      43.0   
2     3_hydroxybutyric_acid   3-Hydroxybutyric acid  HMDB0000011     441.0   
3        3_phosphoglycerate  3-Phosphoglyceric acid  HMDB0000807     724.0   
4                   adenine                 Adenine  HMDB0000034     190.0   
..                      ...                     ...          ...       ...   
129             Unknown_047                     NaN          NaN       NaN   
130             Unknown_048                     NaN          NaN       NaN   
131             Unknown_049                     NaN          NaN       NaN   
132             Unknown_050                     NaN          NaN       NaN   
133             Unknown_051                     NaN          NaN       NaN   

       ChEBI    KEGG  METLIN                 SMILES  Comment  


In [27]:
# How many metabolites_hmdb are in the interactions file?
metabolites_in_interactions = interactions['HMDB'].isin(metabolites_hmdb)
print(f'{metabolites_in_interactions.sum()} metabolites in the interactions file are in the dataset')
# how many metabolites in the dataset are in the interactions file?
metabolites_in_dataset = pd.Series(list(metabolites_hmdb)).isin(interactions['HMDB'])
print(f'{metabolites_in_dataset.sum()} metabolites in the dataset are in the interactions file')

74 metabolites in the interactions file are in the dataset
74 metabolites in the dataset are in the interactions file


# Plot shortest paths between metabolite regulators and module genes in the LemonIte PKN

In [28]:
# Read networks
PKN = pd.read_csv(PKNnetwork_file, sep='\t', header=0)
# split Node1 on '_' and keep the last part
PKN['Node1'] = PKN['Node1'].str.split('_').str[-1]
print(PKN)

lemonnetwork = pd.read_csv(lemonnetwork_file, sep='\t', header=0)
print(lemonnetwork)
# Extract nodes from both networks
lemonnetwork_nodes = list(set(lemonnetwork['Regulator'].tolist() + lemonnetwork['Target'].tolist()))
PKN_nodes = list(set(PKN['Node1'].tolist() + PKN['Node2'].tolist()))

# create name_to_hmdb mapping for metabolites
name_to_hmdb = pd.read_csv(annotated_mets, sep=',')
name_to_hmdb = name_to_hmdb.set_index('Query')['HMDB'].dropna().to_dict()
print(name_to_hmdb)


               Node1     Node2     Source             Type
0            CYP26B1   ALDH1A2     STRING              PPI
1            CYP26B1   ALDH1A3     STRING              PPI
2            CYP26B1   ALDH1A1     STRING              PPI
3            CYP26B1    CRABP1     STRING              PPI
4            CYP26B1    CYP2B6     STRING              PPI
...              ...       ...        ...              ...
2558076  HMDB0304951    MOGAT2  UniProtKB  metabolite-gene
2558077  HMDB0304951       DC5  UniProtKB  metabolite-gene
2558078  HMDB0304951   DGAT2L5  UniProtKB  metabolite-gene
2558079  HMDB0304951    ABHD12  UniProtKB  metabolite-gene
2558080  HMDB0304951  C20orf22  UniProtKB  metabolite-gene

[2558081 rows x 4 columns]
             Regulator  Target      Score  Lemon_module        Type
0                 EMX1  VSTM2L  70.187232             0     TF-gene
1                 EMX1  GABRA5  70.187232             0     TF-gene
2                 EMX1    SYT4  70.187232             0     

In [29]:
# get unique values from Type column
types = PKN['Source'].unique()
print(types)

['STRING' 'BioGRID_genes' 'HuRI' 'BioGRID' 'UniProtKB' 'Human1_GEM_dist1'
 'Human1_GEM_dist2' 'MetalinksDB' 'LINCS' 'chEMBL' 'IntAct' 'STITCH']


In [30]:
# Map metabolite names in 'Regulator' column to HMDB ids
lemonnetwork['Regulator'] = lemonnetwork['Regulator'].map(name_to_hmdb) # Only rows involving a metabolite as regulator will be mapped, other will be set to nan
# drop rows with nan values in 'Regulator' column
lemonnetwork = lemonnetwork.dropna(subset=['Regulator'])
print(lemonnetwork)

         Regulator  Target      Score  Lemon_module             Type
9969   HMDB0000562  VSTM2L  64.743192             0  Metabolite-gene
9970   HMDB0000562  GABRA5  64.743192             0  Metabolite-gene
9971   HMDB0000562    SYT4  64.743192             0  Metabolite-gene
9972   HMDB0000562   CREG2  64.743192             0  Metabolite-gene
9973   HMDB0000562  SYNGR3  64.743192             0  Metabolite-gene
...            ...     ...        ...           ...              ...
14982  HMDB0000192   NXPH3  13.740352            62  Metabolite-gene
14983  HMDB0000192  COL9A1  13.740352            62  Metabolite-gene
14984  HMDB0000192    GAD1  13.740352            62  Metabolite-gene
14985  HMDB0000192  FRMPD1  13.740352            62  Metabolite-gene
14986  HMDB0000192  SMIM18  13.740352            62  Metabolite-gene

[4183 rows x 5 columns]


In [31]:
# remove nodes from lemonnetwork that are not in the PKN
lemonnetwork = lemonnetwork[lemonnetwork['Regulator'].isin(PKN_nodes) & lemonnetwork['Target'].isin(PKN_nodes)]
lemonnetwork
# how many Metabolite-gene interactions remain in lemonnetwork?
#lemonnetwork_metabolites = lemonnetwork[lemonnetwork['Type'] == 'Metabolite-gene']
print(f'The LemonIte network contains {lemonnetwork_metabolites.shape[0]} Metabolite-gene interactions')

The LemonIte network contains 5018 Metabolite-gene interactions


In [32]:
PKN.nx = nx.from_pandas_edgelist(PKN, 'Node1', 'Node2')
# How many nodes and edges does the PKN contain?
print(f'The PKN contains {PKN.nx.number_of_nodes()} nodes and {PKN.nx.number_of_edges()} edges')
lemonnetwork.nx = nx.from_pandas_edgelist(lemonnetwork, 'Regulator', 'Target')
# How many nodes and edges does the LemonIte network contain?
print(f'The LemonIte network contains {lemonnetwork.nx.number_of_nodes()} nodes and {lemonnetwork.nx.number_of_edges()} edges')

The PKN contains 41137 nodes and 1955213 edges
The LemonIte network contains 1821 nodes and 3975 edges


/tmp/ipykernel_78145/162738182.py:1: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  PKN.nx = nx.from_pandas_edgelist(PKN, 'Node1', 'Node2')
/tmp/ipykernel_78145/162738182.py:4: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  lemonnetwork.nx = nx.from_pandas_edgelist(lemonnetwork, 'Regulator', 'Target')


In [33]:
lemonnetwork

,Regulator,Target,Score,Lemon_module,Type
9969,HMDB0000562,VSTM2L,64.743192,0,Metabolite-gene
9970,HMDB0000562,GABRA5,64.743192,0,Metabolite-gene
9971,HMDB0000562,SYT4,64.743192,0,Metabolite-gene
9972,HMDB0000562,CREG2,64.743192,0,Metabolite-gene
9973,HMDB0000562,SYNGR3,64.743192,0,Metabolite-gene
...,...,...,...,...,...
14981,HMDB0000192,TSPAN11,13.740352,62,Metabolite-gene
14982,HMDB0000192,NXPH3,13.740352,62,Metabolite-gene
14983,HMDB0000192,COL9A1,13.740352,62,Metabolite-gene
14984,HMDB0000192,GAD1,13.740352,62,Metabolite-gene


# Plot some shortest paths and their neighborhoods

In [34]:
def create_edge_legend_by_category(ax, edge_styles, edge_info):
    """Create a legend showing edge categories present in the network"""
    from matplotlib.lines import Line2D
    from matplotlib.patches import FancyArrowPatch
    
    # Get unique categories in this network
    present_categories = set()
    for info in edge_info.values():
        present_categories.add(info.get('category', 'Unknown'))
    
    # Create legend elements for all categories in edge_styles
    legend_elements = []
    for category in sorted(edge_styles.keys()):
        style = edge_styles[category]
        
        if category == 'Metabolic_pathway':
            # For Metabolic_pathway, show a line with a dot marker at the end
            legend_elements.append(
                Line2D([0, 1], [0, 0], 
                      marker='o', markersize=6, markerfacecolor=style['color'],
                      markeredgecolor='black', markeredgewidth=0.5,
                      color=style['color'], linewidth=style['width'],
                      linestyle=style['style'], alpha=style['alpha'],
                      label=style['label'])
            )
        elif category == 'Causal':
            # Arrow marker for Causal - use FancyArrowPatch to match drawing
            legend_elements.append(
                FancyArrowPatch((0, 0), (1, 0),
                              arrowstyle='-|>',
                              mutation_scale=15,
                              color=style['color'],
                              linewidth=style['width'],
                              alpha=style['alpha'],
                              label=style['label'])
            )
        else:
            # Normal line for others (PPI, Ambiguous, Unknown)
            legend_elements.append(
                Line2D([0, 1], [0, 0], 
                      color=style['color'], 
                      linewidth=style['width'],
                      linestyle=style['style'],
                      alpha=style['alpha'],
                      label=style['label'])
            )
    
    # Add legend if there are elements
    if legend_elements:
        return legend_elements
    return []

In [35]:
########################################################################################################################################
# Helper functions for categorizing and styling edges based on interaction type and source
########################################################################################################################################

def get_edge_type_category(source, edge_type):
    """
    Categorize edge sources into interaction types based on the edge type (metabolite-gene or PPI)
    
    Parameters:
    -----------
    source : str
        The source database (e.g., 'LINCS', 'STRING', 'BioGRID')
    edge_type : str
        Type of edge - either 'metabolite-gene' or 'PPI'
    
    Returns:
    --------
    str : Interaction category
    """
    
    if edge_type == 'metabolite-gene':
        # Categorization for metabolite-gene interactions only
        causal_sources = ['LINCS', 'chEMBL']
        metabolic_sources = ['Human1_GEM_dist1', 'Human1_GEM_dist2']
        
        if source in causal_sources:
            return 'Causal'
        elif source in metabolic_sources:
            return 'Metabolic_pathway'
        else:
            return 'Ambiguous'
    
    elif edge_type == 'PPI':
        # All PPI interactions are simply categorized as PPI (shown as gray)
        return 'PPI'
    
    else:
        return 'PPI'


def get_edge_style_mapping():
    """
    Define visual styles for each interaction category
    Metabolite-gene interactions have 3 types (Causal, Metabolic_pathway, Ambiguous)
    PPI interactions are all shown as gray lines
    """
    return {
        # Metabolite-gene interaction styles - all dark grey, distinguished by markers
        'Causal': {
            'color': '#555555',  # Dark grey - causal relationships
            'style': 'solid',
            'width': 3.0,
            'alpha': 0.9,
            'label': 'Causal (LINCS/chEMBL)'
        },
        'Metabolic_pathway': {
            'color': '#555555',  # Dark grey - metabolic pathways
            'style': 'solid',
            'width': 2.5,
            'alpha': 0.85,
            'label': 'Metabolic pathway (GEM)'
        },
        'Ambiguous': {
            'color': '#555555',  # Dark grey - ambiguous
            'style': 'solid',
            'width': 2.0,
            'alpha': 0.7,
            'label': 'Ambiguous (other)'
        },
        
        # PPI interaction style (light grey)
        'PPI': {
            'color': '#A9A9A9',  # Darker light grey - protein-protein interactions
            'style': 'solid',
            'width': 1.5,
            'alpha': 0.5,
            'label': 'PPI'
        }
    }


def get_edge_source_and_type(PKN_df, node1, node2):
    """
    Get both the source database and the interaction type for an edge
    
    Returns:
    --------
    tuple : (source, edge_type, category)
    """
    # Check both directions (undirected graph)
    match = PKN_df[((PKN_df['Node1'] == node1) & (PKN_df['Node2'] == node2)) |
                   ((PKN_df['Node1'] == node2) & (PKN_df['Node2'] == node1))]
    
    if not match.empty:
        source = match.iloc[0]['Source']
        edge_type = match.iloc[0]['Type']
        category = get_edge_type_category(source, edge_type)
        return source, edge_type, category
    
    return 'unknown', 'unknown', 'PPI'


def create_edge_legend_by_category(ax, edge_styles, edge_info):
    """Create a legend showing edge categories present in the network"""
    from matplotlib.lines import Line2D
    from matplotlib.patches import FancyArrowPatch
    
    # Get unique categories in this network
    present_categories = set()
    for info in edge_info.values():
        present_categories.add(info.get('category', 'PPI'))
    
    # Create legend elements for all categories in edge_styles
    legend_elements = []
    for category in sorted(edge_styles.keys()):
        style = edge_styles[category]
        
        if category == 'Metabolic_pathway':
            # For Metabolic_pathway, show a line with a dot marker at the end
            legend_elements.append(
                Line2D([0, 1], [0, 0], 
                      marker='o', markersize=6, markerfacecolor=style['color'],
                      markeredgecolor='black', markeredgewidth=0.5,
                      color=style['color'], linewidth=style['width'],
                      linestyle=style['style'], alpha=style['alpha'],
                      label=style['label'])
            )
        elif category == 'Causal':
            # For Causal, show a line with an arrow marker
            legend_elements.append(
                Line2D([0, 1], [0, 0], 
                      marker='>', markersize=8, markerfacecolor=style['color'],
                      markeredgecolor='black', markeredgewidth=0.5,
                      color=style['color'], linewidth=style['width'],
                      linestyle=style['style'], alpha=style['alpha'],
                      label=style['label'])
            )
        else:
            # Normal line for others (PPI, Ambiguous)
            legend_elements.append(
                Line2D([0, 1], [0, 0], 
                      color=style['color'], 
                      linewidth=style['width'],
                      linestyle=style['style'],
                      alpha=style['alpha'],
                      label=style['label'])
            )
    
    # Add legend if there are elements
    if legend_elements:
        return legend_elements
    return []


def create_node_legend(ax, has_metabolites, has_tfs, has_targets, has_bridge):
    """Create a legend showing node colors present in the network"""
    from matplotlib.patches import Patch
    
    legend_elements = []
    
    if has_metabolites:
        legend_elements.append(Patch(facecolor='red', edgecolor='black', label='Metabolites'))
    if has_tfs:
        legend_elements.append(Patch(facecolor='lightgreen', edgecolor='black', label='TFs'))
    if has_targets:
        legend_elements.append(Patch(facecolor='orange', edgecolor='black', label='Targets'))
    if has_bridge:
        legend_elements.append(Patch(facecolor='lightgrey', edgecolor='black', label='Bridge'))
    
    # Return legend elements
    return legend_elements


def print_edge_statistics(edge_info):
    """Print statistics about edge types and categories"""
    from collections import Counter
    
    categories = [info.get('category', 'PPI') for info in edge_info.values()]
    types = [info.get('type', 'unknown') for info in edge_info.values()]
    sources = [info.get('source', 'unknown') for info in edge_info.values()]
    
    print("\n=== Edge Statistics ===")
    print("\nBy Category:")
    for cat, count in Counter(categories).most_common():
        print(f"  {cat}: {count}")
    
    print("\nBy Type:")
    for typ, count in Counter(types).most_common():
        print(f"  {typ}: {count}")
    
    print("\nBy Source:")
    for src, count in Counter(sources).most_common():
        print(f"  {src}: {count}")

In [36]:
def export_to_cytoscape_with_categories(graph, module, hmdb_to_name, TF_regulators, target_genes, edge_info, PKN_df):
    """Export network with edge category information for Cytoscape"""
    import os
    
    cytoscape_dir = "./Networks/cytoscape"
    os.makedirs(cytoscape_dir, exist_ok=True)
    
    # Export edges with full information
    edges_data = []
    for u, v in graph.edges():
        node1 = hmdb_to_name.get(u, u)
        node2 = hmdb_to_name.get(v, v)
        
        info = edge_info.get((u, v)) or edge_info.get((v, u), {})
        source = info.get('source', 'unknown')
        edge_type = info.get('type', 'unknown')
        category = info.get('category', 'PPI')
        
        edges_data.append({
            'Node1': node1,
            'Node2': node2,
            'Source': source,
            'Type': edge_type,
            'Category': category
        })
    
    edges_df = pd.DataFrame(edges_data)
    edges_file = f"{cytoscape_dir}/module_{module}_edges_categorized.tsv"
    edges_df.to_csv(edges_file, sep='\t', index=False)
    
    # Export node attributes
    attributes_data = []
    for node in graph.nodes():
        if node in hmdb_to_name:
            node_type = 'metabolite'
            label = hmdb_to_name[node]
        elif node in TF_regulators:
            node_type = 'TF'
            label = node
        elif node in target_genes:
            node_type = 'module_gene'
            label = node
        else:
            node_type = 'bridge_node'
            label = node
        
        attributes_data.append({
            'Node': label,
            'Label': label,
            'Type': node_type
        })
    
    attributes_df = pd.DataFrame(attributes_data)
    attributes_file = f"{cytoscape_dir}/module_{module}_attributes.tsv"
    attributes_df.to_csv(attributes_file, sep='\t', index=False)
    
    print(f"\nCytoscape files with categorized edges saved for module {module}:")
    print(f"  Edges: {edges_file}")
    print(f"  Attributes: {attributes_file}")


In [37]:
def get_edge_style_mapping():
    """
    Define visual styles for each interaction category
    Metabolite-gene interactions have 3 types (Causal, Metabolic_pathway, Ambiguous)
    PPI interactions are all shown as gray lines
    """
    return {
        # Metabolite-gene interaction styles - all dark grey, distinguished by markers
        'Causal': {
            'color': '#555555',  # Dark grey - causal relationships
            'style': 'solid',
            'width': 3.0,
            'alpha': 0.9,
            'label': 'Causal (LINCS/chEMBL)'
        },
        'Metabolic_pathway': {
            'color': '#555555',  # Dark grey - metabolic pathways
            'style': 'solid',
            'width': 2.5,
            'alpha': 0.85,
            'label': 'Metabolic pathway (GEM)'
        },
        'Ambiguous': {
            'color': '#555555',  # Dark grey - ambiguous
            'style': 'solid',
            'width': 2.0,
            'alpha': 0.7,
            'label': 'Ambiguous (other)'
        },
        
        # PPI interaction style (light grey)
        'PPI': {
            'color': '#A9A9A9',  # Darker light grey - protein-protein interactions
            'style': 'solid',
            'width': 1.5,
            'alpha': 0.5,
            'label': 'PPI'
        },
        
        # Unknown/fallback
        'Unknown': {
            'color': '#555555',  # Dark gray
            'style': 'solid',
            'width': 1.0,
            'alpha': 0.4,
            'label': 'Unknown'
        }
    }

In [38]:
import os
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import FancyArrowPatch
from matplotlib.lines import Line2D
from matplotlib.patches import Patch


def draw_subnetwork(module, target_genes, TF_regulators, metabolite_regulators,
                    PKN_df, PKN_nx, name_to_hmdb):
    """
    Draw subnetwork with clearer arrows, dots, and spacing.
    Uses edge categories and node types for styling.
    """

    # === Setup ===
    hmdb_to_name = {v: k for k, v in name_to_hmdb.items()}
    print(f'Module {module}: {len(metabolite_regulators)} metabolites, '
          f'{len(TF_regulators)} TFs, {len(target_genes)} targets.')

    to_draw = nx.Graph()
    edge_info = {}
    met_regs = []

    # --- STEP 1: Metabolite regulators ---
    for reg in metabolite_regulators:
        if reg not in name_to_hmdb:
            continue
        regulator = name_to_hmdb[reg]
        met_regs.append(regulator)
        if regulator not in PKN_nx:
            continue
        to_draw.add_node(regulator)
        for target in target_genes:
            try:
                if nx.shortest_path_length(PKN_nx, source=regulator, target=target) == 1:
                    to_draw.add_nodes_from([regulator, target])
                    to_draw.add_edge(regulator, target)
                    source, edge_type, category = get_edge_source_and_type(PKN_df, regulator, target)
                    edge_info[(regulator, target)] = {'source': source, 'type': edge_type, 'category': category}
            except (nx.NetworkXNoPath, nx.NodeNotFound):
                continue

    # --- STEP 2: TF regulators (PPI edges) ---
    for TF in TF_regulators:
        for regulator in met_regs:
            try:
                length = nx.shortest_path_length(PKN_nx, source=TF, target=regulator)
                if length <= 2:
                    path = nx.shortest_path(PKN_nx, source=TF, target=regulator)
                    for i in range(len(path) - 1):
                        u, v = path[i], path[i + 1]
                        to_draw.add_edge(u, v)
                        source, edge_type, category = get_edge_source_and_type(PKN_df, u, v)
                        edge_info[(u, v)] = {'source': source, 'type': edge_type, 'category': category}
            except (nx.NetworkXNoPath, nx.NodeNotFound):
                continue

    # --- STEP 3: Add edges among included nodes ---
    subgraph = PKN_nx.subgraph(to_draw.nodes())
    for u, v in subgraph.edges():
        if not to_draw.has_edge(u, v):
            to_draw.add_edge(u, v)
            source, edge_type, category = get_edge_source_and_type(PKN_df, u, v)
            edge_info[(u, v)] = {'source': source, 'type': edge_type, 'category': category}

    to_draw.remove_edges_from(nx.selfloop_edges(to_draw))
    disconnected = [n for n in to_draw if to_draw.degree(n) == 0]
    to_draw.remove_nodes_from(disconnected)

    if not to_draw.nodes:
        print(f"No valid connections for module {module}.")
        return

    # === Layout ===
    pos = nx.spring_layout(to_draw, k=1.8, iterations=200, seed=42)

    # === Node categories ===
    hmdb_nodes = [n for n in to_draw if n in name_to_hmdb.values()]
    tf_nodes = [n for n in to_draw if n in TF_regulators]
    target_nodes = [n for n in to_draw if n in target_genes]
    bridge_nodes = [n for n in to_draw if n not in hmdb_nodes + tf_nodes + target_nodes]

    # === Figure ===
    fig, ax = plt.subplots(figsize=(8, 6))
    edge_styles = get_edge_style_mapping()

    # === Draw nodes ===
    nx.draw_networkx_nodes(
        to_draw, pos,
        nodelist=tf_nodes,
        node_color="lightgreen",
        node_size=800,
        node_shape='o',
        ax=ax
    )
    nx.draw_networkx_nodes(
        to_draw, pos,
        nodelist=target_nodes,
        node_color="orange",
        node_size=800,
        node_shape='o',
        ax=ax
    )
    nx.draw_networkx_nodes(
        to_draw, pos,
        nodelist=bridge_nodes,
        node_color="lightgrey",
        node_size=700,
        node_shape='o',
        ax=ax
    )
    # Metabolites are represented only by their label boxes, not as separate nodes

    # === Draw edges ===
    for category, style in edge_styles.items():
        edges_of_category = [
            (u, v) for (u, v) in to_draw.edges()
            if edge_info.get((u, v), {}).get("category") == category
            or edge_info.get((v, u), {}).get("category") == category
        ]
        if not edges_of_category:
            continue

        if category == "Metabolic_pathway":
            # Draw line ending in a dot at the edge of the target node
            # Orient: metabolite -> target
            for u, v in edges_of_category:
                # Determine direction: metabolite should be source
                if u in hmdb_nodes:
                    source, target = u, v
                elif v in hmdb_nodes:
                    source, target = v, u
                else:
                    # If neither is metabolite, just use u,v
                    source, target = u, v
                
                p1 = np.array(pos[source])
                p2 = np.array(pos[target])
                vec = p2 - p1
                dist = np.linalg.norm(vec)
                if dist == 0:
                    continue
                vec /= dist
                
                # Adjust node radius based on node type
                # Metabolites are squares with size 900, others are circles with size 700-800
                if target in hmdb_nodes:
                    node_radius = 0.065  # Slightly larger for metabolites
                else:
                    node_radius = 0.055
                
                # Stop line at node boundary
                p2_adj = p2 - vec * node_radius
                
                # Draw the line
                ax.plot([p1[0], p2_adj[0]], [p1[1], p2_adj[1]],
                        color=style["color"], linewidth=style["width"], 
                        alpha=style["alpha"], zorder=1)
                
                # Place dot exactly at the node boundary
                dot_point = p2 - vec * node_radius
                ax.plot(dot_point[0], dot_point[1], "o",
                        color=style["color"], markersize=8, 
                        markerfacecolor=style["color"],
                        markeredgecolor='black', markeredgewidth=0.5,
                        alpha=style["alpha"], zorder=3)

        elif category == "Causal":
            # Draw arrows from metabolite to target
            for u, v in edges_of_category:
                # Determine direction: arrow should point from metabolite to target
                start, end = (u, v) if u in hmdb_nodes else (v, u)
                p1 = np.array(pos[start])
                p2 = np.array(pos[end])
                vec = p2 - p1
                dist = np.linalg.norm(vec)
                if dist == 0:
                    continue
                vec /= dist
                
                # Adjust for node sizes
                start_radius = 0.065 if start in hmdb_nodes else 0.055
                end_radius = 0.065 if end in hmdb_nodes else 0.055
                
                p1_adj = p1 + vec * start_radius
                p2_adj = p2 - vec * end_radius
                
                arrow = FancyArrowPatch(
                    p1_adj, p2_adj,
                    arrowstyle='-|>',
                    color=style["color"],
                    linewidth=style["width"],
                    alpha=style["alpha"],
                    mutation_scale=15,
                    connectionstyle='arc3,rad=0',
                    shrinkA=0, shrinkB=0,
                    zorder=2
                )
                ax.add_patch(arrow)

        else:
            # Draw simple edges for PPI, Ambiguous, and Unknown
            nx.draw_networkx_edges(
                to_draw, pos,
                edgelist=edges_of_category,
                edge_color=style["color"],
                style=style["style"],
                width=style["width"],
                alpha=style["alpha"],
                arrows=False,
                ax=ax
            )
    
    # === Labels ===
    metabolite_labels = {n: hmdb_to_name.get(n, n) for n in hmdb_nodes}
    nx.draw_networkx_labels(to_draw, pos, labels=metabolite_labels,
                            font_size=6, font_weight='bold',
                            bbox=dict(boxstyle='round,pad=0.4', fc='red', ec='none', alpha=0.7), ax=ax)
    other_labels = {n: n for n in to_draw if n not in hmdb_nodes}
    nx.draw_networkx_labels(to_draw, pos, labels=other_labels, font_size=5, font_weight='bold', ax=ax)

    # === Legends ===
    edge_legend_elements = create_edge_legend_by_category(ax, edge_styles, edge_info)
    node_legend_elements = create_node_legend(ax,
                                              has_metabolites=bool(hmdb_nodes),
                                              has_tfs=bool(tf_nodes),
                                              has_targets=bool(target_nodes),
                                              has_bridge=bool(bridge_nodes))
    legend_handles = edge_legend_elements + [Line2D([], [], color='none', label='')] + node_legend_elements
    ax.legend(handles=legend_handles, loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=6,
              framealpha=0.9, borderpad=0.5, labelspacing=0.3, handlelength=1.5)

    # === Finalize ===
    plt.title(f'Module {module} - Subnetwork in LemonIte PKN\n'
              f'({len(to_draw.nodes())} nodes, {len(to_draw.edges())} edges)', fontsize=10)
    plt.axis('off')
    plt.tight_layout()

    os.makedirs('./Networks/subnetworks', exist_ok=True)
    plt.savefig(f'./Networks/subnetworks/graph_{module}_categorized_clean.png', dpi=200, bbox_inches='tight')
    plt.close()

    print_edge_statistics(edge_info)
    export_to_cytoscape_with_categories(to_draw, module, hmdb_to_name, TF_regulators, target_genes, edge_info, PKN_df)


# Update the main loop to pass both PKN DataFrame and NetworkX graph
for module in module2mets.keys():
    try:
        draw_subnetwork(
            module, 
            module2genes[module], 
            module2TFs[module], 
            module2mets[module],
            PKN,  # Pass DataFrame
            PKN.nx,  # Pass NetworkX graph
            name_to_hmdb
        )
    except KeyError:
        print(f'Module {module} not found in the mapping file')
        continue
    except Exception as e:
        print(f'An error occurred while processing module {module}: {e}')
        import traceback
        traceback.print_exc()
        continue

Module 0: 2 metabolites, 3 TFs, 88 targets.

=== Edge Statistics ===

By Category:
  PPI: 11
  Metabolic_pathway: 10

By Type:
  PPI: 11
  metabolite-gene: 10

By Source:
  STRING: 11
  Human1_GEM_dist2: 8
  Human1_GEM_dist1: 2

Cytoscape files with categorized edges saved for module 0:
  Edges: ./Networks/cytoscape/module_0_edges_categorized.tsv
  Attributes: ./Networks/cytoscape/module_0_attributes.tsv
Module 1: 4 metabolites, 5 TFs, 71 targets.

=== Edge Statistics ===

By Category:
  PPI: 3
  Ambiguous: 2
  Metabolic_pathway: 1

By Type:
  PPI: 3
  metabolite-gene: 3

By Source:
  UniProtKB: 2
  STRING: 2
  BioGRID_genes: 1
  Human1_GEM_dist2: 1

Cytoscape files with categorized edges saved for module 1:
  Edges: ./Networks/cytoscape/module_1_edges_categorized.tsv
  Attributes: ./Networks/cytoscape/module_1_attributes.tsv
Module 10: 4 metabolites, 4 TFs, 44 targets.

=== Edge Statistics ===

By Category:
  Ambiguous: 2
  PPI: 2
  Metabolic_pathway: 1

By Type:
  metabolite-gene: 3
